In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import gdown
import numpy as np

In [3]:
!gdown --id 1kyCMfQxflrIwYeJnc0MprYEdre7Qu83k -O X_train.npy

!gdown --id 1e3IYJuYUEHrxOPayxP8PlZ45YcK3vip4 -O X_test.npy

!gdown --id 1mF6szu7juZ5n75xOix5CfwwpbpklJIDj -O X_val.npy

!gdown --id 1WwnlFTAZ36W-A-rrvOnq0KzFusdFbYd3 -O y_train.npy

!gdown --id 1qL6KE1XlJmF6epaKzoETb-nOiesZsr39 -O y_test.npy

!gdown --id 1sU9NmM_-qLERVlf02_03Q0yXi3QVg6AP -O y_val.npy

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1kyCMfQxflrIwYeJnc0MprYEdre7Qu83k
From (redirected): https://drive.google.com/uc?id=1kyCMfQxflrIwYeJnc0MprYEdre7Qu83k&confirm=t&uuid=f5e84301-ade1-475b-8e78-3b87ec509080
To: /content/X_train.npy
100% 2.19G/2.19G [00:52<00:00, 41.5MB/s]
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1e3IYJuYUEHrxOPayxP8PlZ45YcK3vip4
From (redirected): https://drive.google.com/uc?id=1e3IYJuYUEHrxOPayxP8PlZ45YcK3vip4&confirm=t&uuid=8ec0e8b2-83e3-4614-a8a1-98394b77435e
To: /content/X_test.npy
10

In [4]:
X_train = np.load('X_train.npy')
y_train = np.load('y_train.npy')

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)

X_train shape: (57788, 256, 37)
y_train shape: (57788,)


In [5]:
X_test = np.load('X_test.npy')
y_test = np.load('y_test.npy')

print('X_test shape:', X_test.shape)
print('y_test shape:', y_test.shape)

X_test shape: (5000, 256, 37)
y_test shape: (5000,)


In [6]:
X_val = np.load('X_val.npy')
y_val = np.load('y_val.npy')

print('X_val shape:', X_val.shape)
print('y_val shape:', y_val.shape)

X_val shape: (5000, 256, 37)
y_val shape: (5000,)


In [7]:
print("Train OCD:", sum(y_train == 1))
print("Train Non-OCD:", sum(y_train == 0))

Train OCD: 20000
Train Non-OCD: 37788


In [8]:
X_ocd = X_train[y_train == 1]
y_ocd = y_train[y_train == 1]

print("OCD samples:", X_ocd.shape)

OCD samples: (20000, 256, 37)


In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional

model = Sequential()

model.add(Bidirectional(LSTM(64, return_sequences=True), input_shape=(256, 37)))
model.add(Dropout(0.3))

model.add(Bidirectional(LSTM(32)))
model.add(Dropout(0.3))

model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [10]:
X_train = X_train.astype('float32')
X_val = X_val.astype('float32')
X_test = X_test.astype('float32')

y_train = y_train.astype('int32')
y_val = y_val.astype('int32')
y_test = y_test.astype('int32')

In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 256, 128)       │        52,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 94,497 (369.13 KB)

 Trainable params: 94,497 (369.13 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [13]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=25,
    batch_size=64,
    callbacks=[early_stopping]
)

Epoch 1/25
903/903 ━━━━━━━━━━━━━━━━━━━━ 43s 41ms/step - accuracy: 0.6541 - loss: 0.6395 - val_accuracy: 0.9258 - val_loss: 0.4493
Epoch 2/25
903/903 ━━━━━━━━━━━━━━━━━━━━ 35s 38ms/step - accuracy: 0.7405 - loss: 0.5244 - val_accuracy: 0.8886 - val_loss: 0.3193
Epoch 3/25
903/903 ━━━━━━━━━━━━━━━━━━━━ 36s 40ms/step - accuracy: 0.8729 - loss: 0.3063 - val_accuracy: 0.8778 - val_loss: 0.3595
Epoch 4/25
903/903 ━━━━━━━━━━━━━━━━━━━━ 35s 38ms/step - accuracy: 0.9320 - loss: 0.1795 - val_accuracy: 0.8838 - val_loss: 0.4285
Epoch 5/25
903/903 ━━━━━━━━━━━━━━━━━━━━ 41s 38ms/step - accuracy: 0.9564 - loss: 0.1199 - val_accuracy: 0.8842 - val_loss: 0.4724
Epoch 6/25
903/903 ━━━━━━━━━━━━━━━━━━━━ 36s 40ms/step - accuracy: 0.9688 - loss: 0.0849 - val_accuracy: 0.8866 - val_loss: 0.5272
Epoch 7/25
903/903 ━━━━━━━━━━━━━━━━━━━━ 35s 39ms/step - accuracy: 0.9751 - loss: 0.0692 - val_accuracy: 0.8852 - val_loss: 0.5410


In [14]:
test_loss, test_acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", test_acc)
print("Test Loss:", test_loss)

157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.8884 - loss: 0.3216
Test Accuracy: 0.8884000182151794
Test Loss: 0.3215870261192322


In [15]:
print("Training Accuracy:", history.history['accuracy'][-1])

Training Accuracy: 0.9750986099243164


In [18]:
from sklearn.metrics import classification_report

for t in [0.5, 0.4, 0.3, 0.2]:
    y_pred = (model.predict(X_test) > t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_test, y_pred))

157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step

Threshold: 0.5
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      4724
           1       0.05      0.06      0.06       276

    accuracy                           0.89      5000
   macro avg       0.50      0.50      0.50      5000
weighted avg       0.90      0.89      0.89      5000

157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step

Threshold: 0.4
              precision    recall  f1-score   support

           0       0.94      0.90      0.92      4724
           1       0.05      0.10      0.07       276

    accuracy                           0.86      5000
   macro avg       0.50      0.50      0.50      5000
weighted avg       0.90      0.86      0.88      5000

157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step

Threshold: 0.3
              precision    recall  f1-score   support

           0       0.94      0.84      0.89      4724
           1       0.05      0.14      0.07       276

    accu